# Uji perbaikan versi 3: chatbot Aiko (`rag_core.py`) + pengaman
**UAS Natural Language Processing, Kelompok B, lanjutan Progress 2 bagian 4**

Hasil sejauh ini pada 15 pertanyaan yang sama (cek manual):

| Versi | Halusinasi | Ditolak padahal bisa dijawab |
|---|---|---|
| 1. Chatbot asli (`rag.ipynb`) | 13 | 0 |
| 2. Prompt sangat ketat (`uji_perbaikan.ipynb`) | 2 | 4 |

Versi 1 terlalu bebas (mengarang), versi 2 terlalu takut (menolak). Versi 3 mencoba menekan keduanya dengan **tidak menyerahkan semua keputusan ke prompt**. Chatbot yang dipakai adalah **`rag_core.py` buatan Aiko** dari branch `gaby/progress2-eval-ui`, supaya hasilnya relevan untuk chatbot yang akan dipakai tim. Kode siapa pun **tidak diubah**: tambahan dibuat di notebook ini.

**Empat tambahan di versi 3:**
1. **Batas relevansi (satpam di pintu):** kalau dokumen teratas terlalu tidak mirip dengan pertanyaan, kode langsung menolak tanpa memanggil LLM.
2. **Potongan artikel yang relevan (chunking):** artikel tidak lagi dipotong 1.500 karakter pertama. Artikel dipecah jadi potongan, lalu potongan yang paling mirip dengan pertanyaan yang dikirim ke LLM, dari 5 artikel teratas.
3. **Prompt "ketat" milik Aiko** (jawab hanya dari dokumen, sebut pasal hanya jika tertulis) ditambah dua aturan: kalimat tolak baku untuk pertanyaan yang sama sekali tidak dibahas, dan tanda **[Dok n]** di setiap pasal atau angka.
4. **Cek klaim otomatis (guru yang mencocokkan):** setelah jawaban keluar, kode mencari semua nomor pasal, UU/PP, nominal Rp, dan durasi (hari/bulan/tahun). Kalau ada yang **tidak ditemukan di teks lengkap artikel**, kalimat yang memuatnya dihapus.

**Syarat:** `uji_luar_dataset.ipynb` sudah dijalankan (hasilnya ada di folder Drive `uas_nlp_uji_luar_dataset_15`). Hasil `uji_perbaikan.ipynb` dipakai juga kalau ada.

**Kuota:** hasil rewrite versi lama dipakai ulang. Per pertanyaan: 1 panggilan jawaban + 1 panggilan juri. Pertanyaan yang ditolak otomatis atau dijawab dengan kalimat tolak baku tidak memanggil juri. Perkiraan sekitar 20 panggilan.

**Sebelum mulai:** Runtime → Change runtime type → **T4 GPU**, dan Colab Secrets `OPENROUTER_API_KEY` (Akses notebook menyala).

## 1. Hubungkan Google Drive, ambil repo (branch Aiko), dan install dependensi

In [1]:
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
CKPT_DIR = "/content/drive/MyDrive/uas_nlp_uji_luar_dataset_15"
assert os.path.exists(os.path.join(CKPT_DIR, "progress_tahap1.json")), \
    "Hasil versi lama tidak ditemukan. Jalankan dulu uji_luar_dataset.ipynb sampai selesai."

REPO_DIR = "/content/UAS-Natural-Language-Processing"
BRANCH = "gaby/progress2-eval-ui"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/sammdavid/UAS-Natural-Language-Processing.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "-q", "origin"], check=True)
ada_branch = subprocess.run(["git", "rev-parse", "--verify", "-q", f"origin/{BRANCH}"], capture_output=True).returncode == 0
subprocess.run(["git", "checkout", "-q", BRANCH if ada_branch else "main"], check=True)
if not ada_branch:
    subprocess.run(["git", "pull", "-q"], check=True)
assert os.path.exists("rag_core.py"), (
    "rag_core.py tidak ada. Branch Aiko mungkin sudah dihapus tetapi belum di-merge ke main. Cek ke tim.")
print("Branch dipakai:", BRANCH if ada_branch else "main (branch Aiko sudah di-merge)")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "rank-bm25",
                "sentence-transformers", "requests", "openpyxl", "python-dotenv"], check=True)

Mounted at /content/drive
Branch dipakai: gaby/progress2-eval-ui


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'faiss-cpu', 'rank-bm25', 'sentence-transformers', 'requests', 'openpyxl', 'python-dotenv'], returncode=0)

## 2. API key dan klien LLM
Key dibaca dari Colab Secrets lalu dipasang sebagai environment variable, karena `rag_core.py` membacanya dari situ.

Untuk memanggil LLM, notebook ini memakai `call_llm` sendiri (payload dan daftar model sama dengan `rag_core`), karena perlu membedakan **kuota harian habis** (notebook berhenti dan progres tersimpan) dari gangguan sesaat (dicoba ulang).

In [2]:
import json, re, time, requests
import numpy as np
import pandas as pd
from google.colab import userdata
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
os.environ["RAG_DATA_DIR"] = REPO_DIR
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import rag_core

class KuotaHabis(Exception):
    pass

def call_llm(messages, temperature=0.2, max_tokens=1000, retries=3):
    headers = {"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}", "Content-Type": "application/json",
               "HTTP-Referer": "https://github.com/sammdavid/UAS-Natural-Language-Processing",
               "X-Title": "UAS NLP Hukumonline RAG"}
    payload = {"models": rag_core.DEFAULT_MODELS, "messages": messages, "temperature": temperature,
               "max_tokens": max_tokens, "reasoning": {"enabled": False}}
    last_err = ""
    for attempt in range(retries):
        try:
            r = requests.post(rag_core.OPENROUTER_URL, headers=headers, json=payload, timeout=90)
            data = r.json()
            if r.status_code == 200 and "error" not in data:
                msg = data.get("choices", [{}])[0].get("message", {})
                content = msg.get("content") or msg.get("reasoning") or ""
                if content.strip():
                    return content.strip()
                last_err = "respon kosong"
            else:
                err = data.get("error", {}) if isinstance(data.get("error"), dict) else {}
                last_err = f"{r.status_code}: {err.get('message', '')}"
                if r.status_code == 429 and "per-day" in last_err:
                    raise KuotaHabis(last_err)
        except KuotaHabis:
            raise
        except Exception as e:
            last_err = str(e)
        time.sleep(5 * (attempt + 1))
    return f"[Error saat memanggil LLM: {last_err}]"

print("Klien LLM siap. Model:", rag_core.DEFAULT_MODELS)

Klien LLM siap. Model: ['google/gemma-4-26b-a4b-it:free', 'google/gemma-4-31b-it:free', 'nvidia/nemotron-3.5-lightning:free']


## 3. Muat data dan indeks lewat `rag_core`
Sama persis dengan chatbot Aiko: FAISS (makna) + BM25 (kata kunci) + model e5-large.

In [3]:
res = rag_core.load_resources()
docs, vectors, embed_model = res.docs, res.vectors, res.embed_model
print(f"{len(docs)} dokumen, device {res.device}")
print("\nPrompt 'ketat' milik Aiko yang dipakai sebagai dasar:\n")
print(rag_core.SYSTEM_PROMPTS["ketat"])

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

7172 dokumen, device cuda

Prompt 'ketat' milik Aiko yang dipakai sebagai dasar:

Anda adalah asisten penyedia informasi hukum umum berbahasa Indonesia, bukan pengacara.
Jawab HANYA berdasarkan dokumen rujukan yang dilampirkan.

Aturan:
1. Mulai dengan jawaban singkat dan lugas atas pertanyaan.
2. Sebutkan dasar hukum (pasal/UU/peraturan) HANYA jika tertulis di dokumen rujukan; jangan menambah pasal dari ingatan.
3. Jelaskan langkah praktis yang disebut atau tersirat jelas di dokumen.
4. Jika dokumen tidak cukup untuk menjawab, katakan terus terang bahwa informasinya tidak ditemukan di dokumen rujukan.
5. Tulis ringkas (maksimal sekitar 300 kata) dan jangan mengulang isi dokumen kata per kata.
6. Ingatkan bahwa ini informasi umum, bukan nasihat hukum, dan aturan bisa berubah.


## 4. Komponen versi 3
- `cari_dokumen`: hybrid search (dense + BM25 + RRF) dengan pertanyaan asli **dan** hasil rewrite, mengambil **5 artikel**. Fungsi peringkatnya memakai `rag_core.dense_ranking` dan `rag_core.sparse_ranking`.
- `skor_relevansi`: kemiripan makna (cosine e5) antara pertanyaan asli dan 3 artikel teratas. Cara hitungnya sama dengan `uji_perbaikan.ipynb`, jadi angkanya bisa dibandingkan.
- `pilih_potongan`: setiap artikel dipecah jadi potongan sekitar 800 karakter (bagian iklan dan "KLINIK TERKAIT" di akhir artikel dibuang), lalu diambil potongan yang paling mirip dengan pertanyaan.
- `cek_klaim` dan `bersihkan_jawaban`: mencocokkan pasal, UU/PP, Rp, dan durasi di jawaban dengan **teks lengkap** 5 artikel; kalimat yang memuat klaim yang tidak ditemukan dihapus.

In [4]:
KALIMAT_TOLAK = "Maaf, sumber kami belum membahas pertanyaan ini."
JUMLAH_ARTIKEL = 5
JUMLAH_POTONGAN = 6
PANJANG_POTONGAN = 800

def cari_dokumen(queries, top_k=JUMLAH_ARTIKEL, pool=30, rrf_k=60):
    rrf = {}
    for q in queries:
        dense_ids, _ = rag_core.dense_ranking(q, pool)
        sparse_ids, _ = rag_core.sparse_ranking(q, pool)
        for ids in (dense_ids, sparse_ids):
            for rank, i in enumerate(ids):
                rrf[i] = rrf.get(i, 0.0) + 1.0 / (rrf_k + rank + 1)
    ranked = sorted(rrf.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [dict(docs[i], doc_idx=i) for i, _ in ranked]

def encode(teks, awalan):
    return embed_model.encode([f"{awalan}: {t}" for t in teks], normalize_embeddings=True,
                              convert_to_numpy=True, batch_size=32)

def skor_relevansi(pertanyaan, artikel):
    q = encode([pertanyaan], "query")[0]
    return max(float(np.dot(vectors[d["doc_idx"]], q)) for d in artikel[:3])

BULAN = r"(?:Jan|Feb|Mar|Apr|Mei|Jun|Jul|Agu|Agt|Ags|Sep|Okt|Nov|Des)\w*"
def isi_bersih(teks):
    """Buang bagian template Hukumonline: daftar KLINIK TERKAIT, info pemutakhiran, penyangkalan, iklan, dan Tags."""
    t = teks.replace("\xa0", " ")
    t = t.replace("ULASAN LENGKAP", " ").replace("Terima kasih atas pertanyaan Anda.", " ")
    t = re.sub(r"KLINIK TERKAIT(?:.{0,200}?\d{1,2} " + BULAN + r" \d{4}){1,6}", " ", t)
    t = re.sub(r"Artikel di bawah ini adalah pemutakhiran.{0,400}?\d{4}\.", " ", t)
    t = re.sub(r"Seluruh informasi hukum yang ada di Klinik.{0,400}?Mitra Justika\.", " ", t)
    t = re.sub(r"Belajar Hukum Secara Online.{0,200}?Lihat Semua Kelas", " ", t)
    i = t.rfind("Tags")
    if i > len(t) * 0.7:
        t = t[:i]
    return re.sub(r"\s+", " ", t).strip(" .")

def potong_artikel(teks, panjang=PANJANG_POTONGAN, geser=600):
    teks = isi_bersih(teks)
    potongan, mulai = [], 0
    while mulai < len(teks):
        akhir = min(len(teks), mulai + panjang)
        if akhir < len(teks):
            spasi = teks.rfind(" ", mulai + geser, akhir)
            akhir = spasi if spasi > 0 else akhir
        potongan.append(teks[mulai:akhir].strip())
        if akhir >= len(teks):
            break
        mulai = max(akhir - (panjang - geser), mulai + 1)
    return potongan

def pilih_potongan(pertanyaan, artikel, n=JUMLAH_POTONGAN, maks_per_artikel=3):
    calon = [(i, p) for i, d in enumerate(artikel) for p in potong_artikel(d.get("answer", ""))]
    q = encode([pertanyaan], "query")[0]
    skor = encode([p for _, p in calon], "passage") @ q
    terpilih, hitung = [], {}
    for k in np.argsort(-skor):
        i, p = calon[k]
        if hitung.get(i, 0) < maks_per_artikel:
            terpilih.append((i, p, float(skor[k])))
            hitung[i] = hitung.get(i, 0) + 1
        if len(terpilih) == n:
            break
    return terpilih

# ---------- cek klaim otomatis ----------
ANGKA_KATA = r"\s*\([a-z ]+\)"
POLA = {
    "pasal": re.compile(r"\bpasal\s+(\d+[a-z]?)", re.I),
    "uu": re.compile(r"\b(?:uu|undang-undang|pp|peraturan pemerintah|perpres|permen\w*|perma|perda|kepmen\w*|keppres)\b\.?\s*(?:no\.?|nomor)?\s*(\d+)\s*(?:/|tahun)\s*(\d{4})", re.I),
    "rp": re.compile(r"rp\.?\s?[\d\.,]+(?:\s?(?:ribu|juta|miliar|triliun))?", re.I),
    "durasi": re.compile(r"\b(\d+)(?:\s*\([a-z ]+\))?\s*(hari|bulan|tahun|minggu|jam)\b(?!\s*\d{4})", re.I),
}

def rupiah(s):
    m = re.match(r"rp\.?([\d\.,]+)(ribu|juta|miliar|triliun)?", s.lower().replace(" ", ""))
    if not m:
        return None
    angka = m.group(1).rstrip(".,")
    if m.group(2):
        kali = {"ribu": 1e3, "juta": 1e6, "miliar": 1e9, "triliun": 1e12}[m.group(2)]
        try:
            return int(float(angka.replace(".", "").replace(",", ".")) * kali)
        except ValueError:
            return None
    angka = re.split(r",\d{2}$", angka)[0].replace(".", "").replace(",", "")
    return int(angka) if angka.isdigit() else None

def ekstrak_klaim(teks):
    out = [("pasal", m.group(0), m.group(1).lower()) for m in POLA["pasal"].finditer(teks)]
    out += [("uu", m.group(0), (m.group(1), m.group(2))) for m in POLA["uu"].finditer(teks)]
    out += [("rp", m.group(0), rupiah(m.group(0))) for m in POLA["rp"].finditer(teks) if rupiah(m.group(0))]
    out += [("durasi", m.group(0), (m.group(1), m.group(2).lower())) for m in POLA["durasi"].finditer(teks)]
    return out

def indeks_artikel(teks):
    t = re.sub(r"\s+", " ", re.sub(ANGKA_KATA, "", teks.lower()))
    return {
        "pasal": {x.lower() for x in POLA["pasal"].findall(t)},
        "uu": set(POLA["uu"].findall(t)) | set(re.findall(r"\b(\d+)/(\d{4})\b", t)),
        "rp": {v for v in (rupiah(x) for x in POLA["rp"].findall(t)) if v},
        "durasi": {(a, b.lower()) for a, b in re.findall(r"\b(\d+)\s*(hari|bulan|tahun|minggu|jam)\b", t)},
        "tahun": set(re.findall(r"\b(1[89]\d\d|20\d\d)\b", t)),
    }

def cek_klaim(jawaban, teks_artikel):
    """Kembalikan daftar klaim (pasal/UU/Rp/durasi) di jawaban yang TIDAK ada di teks artikel."""
    idx = indeks_artikel(teks_artikel)
    tidak_ada = []
    for jenis, asli, kunci in ekstrak_klaim(jawaban):
        if jenis == "durasi" and kunci[1] == "tahun" and kunci[0] in idx["tahun"]:
            continue  # "tahun 2023" itu tahun, bukan durasi
        if kunci not in idx[jenis]:
            tidak_ada.append(asli.strip())
    return sorted(set(tidak_ada))

def bersihkan_jawaban(jawaban, klaim_tidak_ada):
    """Hapus kalimat yang memuat klaim tidak ditemukan. Kalau sisa isinya terlalu sedikit, ganti dengan kalimat tolak."""
    if not klaim_tidak_ada:
        return jawaban
    baris_baru = []
    for baris in jawaban.split("\n"):
        kalimat = re.split(r"(?<=[.!?])\s+", baris)
        sisa = [k for k in kalimat if not any(c.lower() in k.lower() for c in klaim_tidak_ada)]
        if len(sisa) == len(kalimat) or (sisa and re.sub(r"[\W_]", "", " ".join(sisa))):
            baris_baru.append(" ".join(sisa))
    hasil = re.sub(r"\n{3,}", "\n\n", "\n".join(baris_baru)).strip()
    isi = re.sub(r"https?://\S+|\[Dok \d+\]", "", hasil)
    if len(re.sub(r"\W", "", isi)) < 150:
        return KALIMAT_TOLAK
    return hasil + f"\n\n_(Catatan sistem: {len(klaim_tidak_ada)} klaim dihapus karena tidak ditemukan di dokumen rujukan.)_"

print("Komponen versi 3 siap.")

Komponen versi 3 siap.


### Prompt versi 3
Dasarnya prompt **"ketat" milik Aiko, tanpa diubah**. Ditambah tiga aturan di akhir:
- tolak dengan kalimat baku **hanya** kalau dokumen sama sekali tidak terkait atau pertanyaannya bukan soal hukum;
- kalau dokumen membahas kasus yang mirip atau sebagian, tetap jawab dari dokumen dan sebut bagian yang tidak tercakup (ini yang membedakan dengan versi 2 yang terlalu ketat);
- beri tanda **[Dok n]** setelah setiap pasal, angka, atau sanksi, supaya cek manual lebih cepat.

In [5]:
TAMBAHAN_V3 = (
    "\n\nAturan tambahan:\n"
    f"7. Jika pertanyaan BUKAN tentang hukum, atau dokumen rujukan SAMA SEKALI tidak terkait dengan pokok pertanyaan, jawab PERSIS: \"{KALIMAT_TOLAK}\" lalu berhenti.\n"
    "8. Jika dokumen membahas kasus yang mirip atau sebagian dari pertanyaan, TETAP jawab berdasarkan dokumen itu, lalu sebutkan bagian pertanyaan yang tidak dibahas dokumen. Jangan menolak hanya karena kasusnya tidak persis sama.\n"
    "9. Setelah setiap pasal, nomor peraturan, angka, atau sanksi, tulis sumbernya dalam format [Dok n]."
)
SYSTEM_V3 = rag_core.SYSTEM_PROMPTS["ketat"] + TAMBAHAN_V3

def rag_chat_v3(pertanyaan, rewritten=None, batas=None):
    queries = [pertanyaan] + ([rewritten] if rewritten and rewritten != pertanyaan else [])
    artikel = cari_dokumen(queries)
    skor = skor_relevansi(pertanyaan, artikel)
    hasil = {"artikel": artikel, "skor": skor, "potongan": [], "jawaban_mentah": "",
             "klaim_tidak_ada": [], "ditolak_otomatis": False}
    if batas is not None and skor < batas:
        hasil.update(jawaban=KALIMAT_TOLAK, ditolak_otomatis=True)
        return hasil
    potongan = pilih_potongan(pertanyaan, artikel)
    hasil["potongan"] = potongan
    blok = []
    for i, d in enumerate(artikel):
        isi = [p for j, p, _ in potongan if j == i]
        if isi:
            blok.append(f"[Dok {len(blok) + 1}]\nJudul: {d['title']}\nRingkasan Masalah: {d['summarize']}\n"
                        "Potongan yang relevan:\n" + "\n...\n".join(isi) + f"\nSumber: {d['source']}")
    user = (f"Pertanyaan Pengguna:\n{pertanyaan}\n\nBerikut adalah Dokumen Rujukan Resmi dari Klinik Hukumonline:\n"
            + "\n---\n".join(blok) + f"\n\n{rag_core.USER_PROMPT_ENDING['ketat']}")
    mentah = call_llm([{"role": "system", "content": SYSTEM_V3}, {"role": "user", "content": user}],
                      temperature=0.1, max_tokens=900)
    hasil["jawaban_mentah"] = mentah
    hasil["konteks"] = user
    if mentah.startswith("[Error"):
        hasil["jawaban"] = mentah
        return hasil
    teks_lengkap = " ".join(d.get("answer", "") for d in artikel)
    hasil["klaim_tidak_ada"] = cek_klaim(mentah, teks_lengkap)
    hasil["jawaban"] = bersihkan_jawaban(mentah, hasil["klaim_tidak_ada"])
    return hasil

print(SYSTEM_V3)

Anda adalah asisten penyedia informasi hukum umum berbahasa Indonesia, bukan pengacara.
Jawab HANYA berdasarkan dokumen rujukan yang dilampirkan.

Aturan:
1. Mulai dengan jawaban singkat dan lugas atas pertanyaan.
2. Sebutkan dasar hukum (pasal/UU/peraturan) HANYA jika tertulis di dokumen rujukan; jangan menambah pasal dari ingatan.
3. Jelaskan langkah praktis yang disebut atau tersirat jelas di dokumen.
4. Jika dokumen tidak cukup untuk menjawab, katakan terus terang bahwa informasinya tidak ditemukan di dokumen rujukan.
5. Tulis ringkas (maksimal sekitar 300 kata) dan jangan mengulang isi dokumen kata per kata.
6. Ingatkan bahwa ini informasi umum, bukan nasihat hukum, dan aturan bisa berubah.

Aturan tambahan:
7. Jika pertanyaan BUKAN tentang hukum, atau dokumen rujukan SAMA SEKALI tidak terkait dengan pokok pertanyaan, jawab PERSIS: "Maaf, sumber kami belum membahas pertanyaan ini." lalu berhenti.
8. Jika dokumen membahas kasus yang mirip atau sebagian dari pertanyaan, TETAP jawab 

## 5. Muat hasil versi 1 dan versi 2 dari Google Drive
Tidak ada panggilan LLM di sini.

In [6]:
lama1 = pd.DataFrame(json.load(open(os.path.join(CKPT_DIR, "progress_tahap1.json"))))
lama2 = pd.DataFrame(json.load(open(os.path.join(CKPT_DIR, "progress_tahap2.json"))))
lama = lama1.merge(lama2, on="no").sort_values("no").reset_index(drop=True)
p_v2 = os.path.join(CKPT_DIR, "progress_perbaikan.json")
if os.path.exists(p_v2):
    v2 = pd.DataFrame(json.load(open(p_v2)))[["no", "label_baru", "jawaban_baru"]]
    lama = lama.merge(v2.rename(columns={"label_baru": "label_v2", "jawaban_baru": "jawaban_v2"}), on="no", how="left")
else:
    lama["label_v2"], lama["jawaban_v2"] = None, None
print(len(lama), "pertanyaan dimuat | hasil versi 2 ada:", os.path.exists(p_v2))
lama[["no", "kelompok", "pertanyaan", "label_jawaban_llm", "label_v2"]]

15 pertanyaan dimuat | hasil versi 2 ada: True


,no,kelompok,pertanyaan,label_jawaban_llm,label_v2
0,1,A,"Gaji lembur saya tidak dibayar kantor, saya ha...",HALUSINASI,HALUSINASI
1,2,A,Bagaimana cara menggugat cerai kalau suami tid...,HALUSINASI,HALUSINASI
2,3,A,Apakah aman membeli tanah yang masih berupa gi...,HALUSINASI,MENOLAK_JUJUR
3,4,A,"Teman pinjam uang lalu kabur, bisa saya lapork...",HALUSINASI,HALUSINASI
4,5,A,"Saya kena PHK tanpa pesangon, apa yang bisa sa...",HALUSINASI,HALUSINASI
5,6,B,Apa itu pidana kerja sosial dalam KUHP baru ya...,HALUSINASI,HALUSINASI
6,7,B,Bagaimana aturan hukum penggunaan AI untuk mem...,HALUSINASI,HALUSINASI
7,8,B,Apakah kripto hasil airdrop wajib dilaporkan d...,HALUSINASI,MENOLAK_JUJUR
8,9,B,Apa sanksi bagi pengemudi mobil listrik yang m...,HALUSINASI,HALUSINASI
9,10,B,Bagaimana status hukum pernikahan yang dilangs...,HALUSINASI,HALUSINASI


## 6. Uji cek klaim otomatis pada jawaban versi 1 (tanpa kuota)
Sebelum dipakai, cek klaim diuji dulu pada jawaban chatbot asli: klaim mana yang akan tertangkap kalau pengaman ini sudah ada sejak awal.

**Cara membaca:** kolom `klaim_tidak_ditemukan` berisi pasal, UU, Rp, atau durasi yang ditulis chatbot tetapi tidak ada di teks lengkap ketiga artikelnya.

**Batasan (tulis di laporan):** cek ini hanya mencocokkan **nomor**. Klaim yang nomornya ada di artikel tetapi dipasangkan secara salah (misalnya bunyi Pasal 362 dikutip sebagai Pasal 476) tidak tertangkap, begitu juga karangan berupa kalimat biasa atau nama lembaga.

In [7]:
judul_ke_teks = {}
for d in docs:
    judul_ke_teks.setdefault(d["title"].strip(), []).append(d.get("answer", ""))
uji_cek = []
for _, r in lama.iterrows():
    teks = " ".join(t for j in str(r["dokumen_dipakai"]).split("|") for t in judul_ke_teks.get(j.strip(), []))
    uji_cek.append({"no": r["no"], "kelompok": r["kelompok"],
                    "klaim_tidak_ditemukan": ", ".join(cek_klaim(str(r["jawaban_chatbot"]), teks)) or "-"})
pd.DataFrame(uji_cek)

,no,kelompok,klaim_tidak_ditemukan
0,1,A,-
1,2,A,"14 hari, Pasal 26, Pasal 35"
2,3,A,6 bulan
3,4,A,-
4,5,A,3 bulan
5,6,B,-
6,7,B,"9 (sembilan) tahun, Pasal 287, Pasal 288, Pasa..."
7,8,B,-
8,9,B,Rp. 500
9,10,B,"Pasal 2, Pasal 40"


## 7. Skor relevansi dan batas (tanpa kuota)
Tabel menunjukkan skor tiap pertanyaan. Di `uji_perbaikan.ipynb`, kelompok C (bukan hukum) mendapat skor **0,77 sampai 0,81**, sedangkan kelompok A dan B **0,82 ke atas**. Karena itu `BATAS_RELEVANSI` diisi **0,815**.

**Catatan jujur untuk laporan:** batas ini ditentukan dari 15 pertanyaan uji yang sama, jadi hasilnya cenderung terlalu optimis. Idealnya batas diuji ulang dengan kumpulan pertanyaan lain. Kalau ingin menguji tanpa batas, isi `None`.

In [8]:
BATAS_RELEVANSI = 0.815   # None = tidak dipakai

skor_rows = []
for _, r in lama.iterrows():
    queries = [r["pertanyaan"]] + ([r["query_rewrite"]] if r["query_rewrite"] != r["pertanyaan"] else [])
    art = cari_dokumen(queries)
    s = skor_relevansi(r["pertanyaan"], art)
    skor_rows.append({"no": r["no"], "kelompok": r["kelompok"], "pertanyaan": r["pertanyaan"][:60],
                      "skor_relevansi": round(s, 4),
                      "ditolak_otomatis": BATAS_RELEVANSI is not None and s < BATAS_RELEVANSI,
                      "artikel_teratas": art[0]["title"][:60]})
skor_df = pd.DataFrame(skor_rows)
print(skor_df.groupby("kelompok")["skor_relevansi"].agg(["min", "mean", "max"]).round(4))
skor_df

             min    mean     max
kelompok                        
A         0.8543  0.8688  0.8847
B         0.8249  0.8393  0.8711
C         0.7748  0.7933  0.8071


,no,kelompok,pertanyaan,skor_relevansi,ditolak_otomatis,artikel_teratas
0,1,A,"Gaji lembur saya tidak dibayar kantor, saya ha...",0.8543,False,Lembur Dianggap Kadaluwarsa
1,2,A,Bagaimana cara menggugat cerai kalau suami tid...,0.8669,False,Bisakah Istri Diam-Diam Menggugat Cerai Suami?
2,3,A,Apakah aman membeli tanah yang masih berupa gi...,0.8847,False,Kiat-kiat Membeli Tanah Girik yang Aman
3,4,A,"Teman pinjam uang lalu kabur, bisa saya lapork...",0.8761,False,Bisakah Pelanggaran Perjanjian Pinjam Pakai Di...
4,5,A,"Saya kena PHK tanpa pesangon, apa yang bisa sa...",0.8622,False,Pesangon Bagi Korban PHK Karena Berakhirnya Pr...
5,6,B,Apa itu pidana kerja sosial dalam KUHP baru ya...,0.8304,False,Bunyi Pasal 252 KUHP Baru tentang Santet
6,7,B,Bagaimana aturan hukum penggunaan AI untuk mem...,0.8711,False,Apa Itu Deepfake Porn dan Jerat Pidana bagi Pe...
7,8,B,Apakah kripto hasil airdrop wajib dilaporkan d...,0.8326,False,3 Langkah Transaksi Aset Kripto Secara Legal d...
8,9,B,Apa sanksi bagi pengemudi mobil listrik yang m...,0.8249,False,Membuktikan Dugaan Pencurian Listrik
9,10,B,Bagaimana status hukum pernikahan yang dilangs...,0.8377,False,Hukumnya Menyembunyikan Status Pernikahan


## 8. Jalankan chatbot versi 3 dan juri
Juri memakai **prompt yang sama** dengan notebook sebelumnya, dan diberi **potongan dokumen yang sama** dengan yang dibaca chatbot. Dua jenis jawaban langsung diberi label `MENOLAK_JUJUR` tanpa memanggil juri: yang ditolak otomatis oleh batas relevansi, dan yang isinya persis kalimat tolak baku (di versi 2, juri keliru menilai jawaban seperti ini sebagai halusinasi).

Progres disimpan ke `progress_v3.json` di folder Drive yang sama. Untuk mengulang dari nol (misalnya setelah mengganti `BATAS_RELEVANSI`), hapus file itu dulu.

In [9]:
def ambil_json(teks):
    m = re.search(r"\{.*\}", teks, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

JURI_JAWABAN = (
    "Anda adalah juri pendeteksi halusinasi untuk chatbot hukum berbasis dokumen. Anda diberi PERTANYAAN, "
    "DOKUMEN RUJUKAN yang dipakai chatbot, dan JAWABAN chatbot. Beri satu label:\n"
    "- BERDASAR_DOKUMEN: klaim penting di jawaban didukung dokumen rujukan.\n"
    "- MENOLAK_JUJUR: chatbot menyatakan dokumen tidak cukup / di luar cakupan dan tidak mengarang.\n"
    "- HALUSINASI: ada klaim penting (pasal, aturan, angka, prosedur) yang tidak ada di dokumen rujukan, "
    "atau chatbot menjawab topik di luar dokumen seolah-olah tahu.\n"
    'Jawab HANYA JSON: {"label": "...", "klaim_tak_didukung": "kutip singkat atau kosong", "alasan": "satu kalimat"}'
)

def juri_jawaban(konteks, jawaban):
    out = call_llm([{"role": "system", "content": JURI_JAWABAN},
                    {"role": "user", "content": f"{konteks}\n\nJAWABAN CHATBOT:\n{jawaban}"}],
                   temperature=0.0, max_tokens=400)
    return ambil_json(out) or {"label": None, "klaim_tak_didukung": "", "alasan": f"Gagal dibaca: {out[:150]}"}

CKPT_V3 = os.path.join(CKPT_DIR, "progress_v3.json")
hasil_v3 = [r for r in (json.load(open(CKPT_V3)) if os.path.exists(CKPT_V3) else []) if r["label_v3"]]
sudah = {r["no"] for r in hasil_v3}
print(f"Melanjutkan dari {len(sudah)} pertanyaan" if sudah else "Mulai dari awal", "| batas relevansi:", BATAS_RELEVANSI)

for _, r in lama.iterrows():
    if r["no"] in sudah:
        continue
    try:
        h = rag_chat_v3(r["pertanyaan"], rewritten=r["query_rewrite"], batas=BATAS_RELEVANSI)
        if h["ditolak_otomatis"]:
            v = {"label": "MENOLAK_JUJUR", "klaim_tak_didukung": "", "alasan": "Ditolak otomatis oleh batas relevansi."}
        elif h["jawaban"].startswith("[Error"):
            v = {"label": None, "klaim_tak_didukung": "", "alasan": h["jawaban"][:150]}
        elif h["jawaban"].strip() == KALIMAT_TOLAK:
            v = {"label": "MENOLAK_JUJUR", "klaim_tak_didukung": "",
                 "alasan": "Jawaban hanya kalimat tolak baku" + (" (setelah klaim dihapus)." if h["klaim_tidak_ada"] else ".")}
        else:
            v = juri_jawaban(h["konteks"], h["jawaban"])
    except KuotaHabis:
        print(f"\nKUOTA HARIAN HABIS di pertanyaan {r['no']}. Progres tersimpan ({len(hasil_v3)} selesai).")
        print("Jalankan lagi setelah kuota reset atau ganti key; yang selesai tidak diulang.")
        break
    hasil_v3.append({
        "no": int(r["no"]), "skor_relevansi": round(h["skor"], 4), "ditolak_otomatis": h["ditolak_otomatis"],
        "artikel_v3": " | ".join(d["title"] for d in h["artikel"]),
        "potongan_dipakai": "\n\n".join(f"[{h['artikel'][j]['title'][:50]}] {p}" for j, p, _ in h["potongan"]),
        "jawaban_mentah_v3": h["jawaban_mentah"], "klaim_dihapus": ", ".join(h["klaim_tidak_ada"]),
        "jawaban_v3": h["jawaban"], "label_v3": v.get("label"),
        "klaim_tak_didukung_v3": v.get("klaim_tak_didukung"), "alasan_v3": v.get("alasan"),
    })
    json.dump(hasil_v3, open(CKPT_V3, "w"), ensure_ascii=False)
    info = f" | klaim dihapus: {', '.join(h['klaim_tidak_ada'])}" if h["klaim_tidak_ada"] else ""
    if not v.get("label"):
        info += f"  <- {v.get('alasan', '')[:120]}"
    print(f"[{int(r['no']):02d}] {r['kelompok']} | v1={r['label_jawaban_llm']} | v2={r['label_v2']} | v3={v.get('label')}{info}")
    time.sleep(2)

if len(hasil_v3) < len(lama):
    raise SystemExit(f"Baru {len(hasil_v3)}/{len(lama)} selesai. Jalankan semua lagi nanti.")

Mulai dari awal | batas relevansi: 0.815
[01] A | v1=HALUSINASI | v2=HALUSINASI | v3=MENOLAK_JUJUR
[02] A | v1=HALUSINASI | v2=HALUSINASI | v3=BERDASAR_DOKUMEN
[03] A | v1=HALUSINASI | v2=MENOLAK_JUJUR | v3=None  <- Gagal dibaca: {"label": "HALUSINASI", "klaim_tak_didukung": "Membeli tanah girik memerlukan pengawasan ketat karena stat
[04] A | v1=HALUSINASI | v2=HALUSINASI | v3=BERDASAR_DOKUMEN
[05] A | v1=HALUSINASI | v2=HALUSINASI | v3=BERDASAR_DOKUMEN
[06] B | v1=HALUSINASI | v2=HALUSINASI | v3=BERDASAR_DOKUMEN
[07] B | v1=HALUSINASI | v2=HALUSINASI | v3=HALUSINASI | klaim dihapus: Undang-Undang Nomor 44 Tahun 2008
[08] B | v1=HALUSINASI | v2=MENOLAK_JUJUR | v3=MENOLAK_JUJUR
[09] B | v1=HALUSINASI | v2=HALUSINASI | v3=HALUSINASI
[10] B | v1=HALUSINASI | v2=HALUSINASI | v3=HALUSINASI
[11] C | v1=HALUSINASI | v2=HALUSINASI | v3=MENOLAK_JUJUR
[12] C | v1=HALUSINASI | v2=HALUSINASI | v3=MENOLAK_JUJUR
[13] C | v1=HALUSINASI | v2=MENOLAK_JUJUR | v3=MENOLAK_JUJUR
[14] C | v1=MENOLAK_JUJUR 

## 9. Perbandingan versi 1, 2, dan 3 (label juri)
Ingat: ini **label juri**, yang di versi sebelumnya terbukti sering keliru. Kesimpulan akhir tetap dari cek manual di Excel.

**Target ideal:** kelompok A dijawab dari dokumen (`BERDASAR_DOKUMEN`), kelompok B dan C ditolak (`MENOLAK_JUJUR`), kecuali kelompok B yang dokumennya memang membahas sebagian (boleh dijawab sebagian).

In [10]:
v3 = pd.DataFrame(hasil_v3)
banding = lama.merge(v3, on="no")
ringkas = pd.DataFrame({
    "versi_1": banding["label_jawaban_llm"].value_counts(),
    "versi_2": banding["label_v2"].value_counts(),
    "versi_3": banding["label_v3"].value_counts(),
}).fillna(0).astype(int)
print(ringkas, "\n")
print(pd.crosstab(banding["kelompok"], banding["label_v3"].fillna("GAGAL")).rename_axis(None, axis=1), "\n")
print("Pertanyaan dengan klaim yang dihapus otomatis:", (banding["klaim_dihapus"] != "").sum())
banding[["no", "kelompok", "pertanyaan", "label_jawaban_llm", "label_v2", "label_v3", "skor_relevansi", "klaim_dihapus"]]

                  versi_1  versi_2  versi_3
BERDASAR_DOKUMEN        0        0        4
HALUSINASI             14       12        3
MENOLAK_JUJUR           1        3        7 

          BERDASAR_DOKUMEN  GAGAL  HALUSINASI  MENOLAK_JUJUR
kelompok                                                    
A                        3      1           0              1
B                        1      0           3              1
C                        0      0           0              5 

Pertanyaan dengan klaim yang dihapus otomatis: 1


,no,kelompok,pertanyaan,label_jawaban_llm,label_v2,label_v3,skor_relevansi,klaim_dihapus
0,1,A,"Gaji lembur saya tidak dibayar kantor, saya ha...",HALUSINASI,HALUSINASI,MENOLAK_JUJUR,0.8543,
1,2,A,Bagaimana cara menggugat cerai kalau suami tid...,HALUSINASI,HALUSINASI,BERDASAR_DOKUMEN,0.8669,
2,3,A,Apakah aman membeli tanah yang masih berupa gi...,HALUSINASI,MENOLAK_JUJUR,None,0.8847,
3,4,A,"Teman pinjam uang lalu kabur, bisa saya lapork...",HALUSINASI,HALUSINASI,BERDASAR_DOKUMEN,0.8761,
4,5,A,"Saya kena PHK tanpa pesangon, apa yang bisa sa...",HALUSINASI,HALUSINASI,BERDASAR_DOKUMEN,0.8622,
5,6,B,Apa itu pidana kerja sosial dalam KUHP baru ya...,HALUSINASI,HALUSINASI,BERDASAR_DOKUMEN,0.8304,
6,7,B,Bagaimana aturan hukum penggunaan AI untuk mem...,HALUSINASI,HALUSINASI,HALUSINASI,0.8711,Undang-Undang Nomor 44 Tahun 2008
7,8,B,Apakah kripto hasil airdrop wajib dilaporkan d...,HALUSINASI,MENOLAK_JUJUR,MENOLAK_JUJUR,0.8326,
8,9,B,Apa sanksi bagi pengemudi mobil listrik yang m...,HALUSINASI,HALUSINASI,HALUSINASI,0.8249,
9,10,B,Bagaimana status hukum pernikahan yang dilangs...,HALUSINASI,HALUSINASI,HALUSINASI,0.8377,


## 10. Simpan Excel untuk cek manual
Kolom penting:
- `potongan_dipakai`: potongan artikel yang benar-benar dibaca chatbot. Cek manual cukup mencocokkan jawaban dengan kolom ini.
- `jawaban_mentah_v3`: jawaban asli LLM sebelum dibersihkan; `klaim_dihapus`: klaim yang dibuang oleh cek otomatis; `jawaban_v3`: jawaban akhir yang dilihat pengguna.
- `cek_manual_v3` dan `catatan`: diisi manual. Nilai `jawaban_v3` dengan label yang sama (BERDASAR_DOKUMEN / MENOLAK_JUJUR / HALUSINASI), dan tulis "seharusnya bisa dijawab" di catatan kalau kelompok A ditolak.

In [11]:
banding["cek_manual_v3"] = ""
banding["catatan"] = ""
kolom = ["no", "kelompok", "pertanyaan", "label_jawaban_llm", "label_v2", "skor_relevansi", "ditolak_otomatis",
         "artikel_v3", "potongan_dipakai", "jawaban_mentah_v3", "klaim_dihapus", "jawaban_v3",
         "label_v3", "klaim_tak_didukung_v3", "alasan_v3", "cek_manual_v3", "catatan"]
banding[kolom].rename(columns={"label_jawaban_llm": "label_v1"}).to_excel("uji_perbaikan_v3.xlsx", index=False)
print("Tersimpan: uji_perbaikan_v3.xlsx")
from google.colab import files
files.download("uji_perbaikan_v3.xlsx")

Tersimpan: uji_perbaikan_v3.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Catatan keterbatasan
- Hanya 15 pertanyaan, dan batas relevansi ditentukan dari pertanyaan yang sama, jadi hasilnya indikatif.
- Cek klaim otomatis hanya mencocokkan nomor pasal, UU/PP, Rp, dan durasi. Salah pasang (nomor benar, isi salah), nama lembaga, dan karangan berupa kalimat biasa tidak tertangkap.
- Juri memakai keluarga model yang sama (Gemma) dan terbukti bisa keliru, jadi kesimpulan akhir memakai cek manual.
- Kalau hasilnya baik, usulan ke tim cukup berupa tambahan di `rag_core.py`: batas relevansi, `pilih_potongan`, aturan tambahan prompt, dan `cek_klaim`.